In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
%%bash
# Upgrade pip
pip install --upgrade pip

# Install causal-conv1d and mamba-ssm
# Note: --no-build-isolation ensures it uses the environment's existing PyTorch & CUDA
pip install causal-conv1d>=1.4.0 --no-build-isolation
pip install mamba-ssm --no-build-isolation

# Install transformers, datasets, and accelerate
pip install transformers datasets accelerate evaluate trl

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print("Visible GPUs:", torch.cuda.device_count())  # Must print: 1

In [ ]:
import torch
import mamba_ssm
import causal_conv1d
print("CUDA Available:", torch.cuda.is_available())
print("Mamba Version:", mamba_ssm.__version__)

In [ ]:
pip install "transformers<=4.46.3"
import os
os._exit(00)

In [ ]:
from transformers import MambaForCausalLM, AutoTokenizer
import torch

model_id = "state-spaces/mamba-130m-hf"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = MambaForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [ ]:
import os
import torch

# 1. Patch the transformers lazy_load_kernel bug
import mamba_ssm
try:
    import mamba_ssm.ops.triton.selective_state_update as ssu
    mamba_ssm.selective_state_update = getattr(ssu, "selective_state_update", None)
except (ImportError, ModuleNotFoundError):
    mamba_ssm.selective_state_update = None

from transformers import AutoTokenizer, MambaForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_id = "state-spaces/mamba-130m-hf"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load Pretrained Mamba Model
model = MambaForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,    # Use float16 for T4 GPU
    device_map={"": 0}            # Force onto the primary GPU
)

model.config.pad_token_id = tokenizer.pad_token_id

In [ ]:
# Load a subset of Alpaca dataset for demonstration
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:5000]")

# Split into 90% train and 10% validation
dataset = dataset.train_test_split(test_size=0.1)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# Format instruction prompts
def format_prompts(batch):
    formatted_texts = []
    for instruction, input_text, output in zip(batch["instruction"], batch["input"], batch["output"]):
        if input_text:
            text = f"Below is an instruction that describes a task, paired with an input that provides further context.\n\n### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"
        else:
            text = f"Below is an instruction that describes a task.\n\n### Instruction:\n{instruction}\n\n### Response:\n{output}"
        formatted_texts.append(text)
    return {"text": formatted_texts}

train_dataset = train_dataset.map(format_prompts, batched=True)
eval_dataset = eval_dataset.map(format_prompts, batched=True)

# Tokenize data
max_seq_length = 512

def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_seq_length,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_train = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(tokenize, batched=True, remove_columns=eval_dataset.column_names)

In [ ]:
from transformers import DataCollatorForSeq2Seq, TrainingArguments, Trainer

# 1. Cast trainable weights to float32 (fixes the unscale error)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()

# 2. Disable use_cache during training
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir="./mamba-fine-tuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,              # 5e-5 is recommended for full fine-tuning
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    fp16=True,                       # Fast FP16 forward pass
    weight_decay=0.01,
    prediction_loss_only=True,       # Avoids MambaCache gather issues during eval
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt"),
)

# Start Fine-Tuning
trainer.train()

In [ ]:
save_path = "./final_mamba_model"

# Save model and tokenizer
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

In [ ]:
trainer.save_model("./mamba-fine-tuned")
tokenizer.save_pretrained("./mamba-fine-tuned")
print("Model saved successfully!")

In [ ]:
clean_model = MambaForCausalLM.from_pretrained(
    "./mamba-fine-tuned",
    torch_dtype=torch.float16,
    device_map="cuda:0"
)

prompt = "Below is an instruction that describes a task.\n\n### Instruction:\nExplain what machine learning is in one simple sentence.\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = clean_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=60,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
# 1. Save weights and tokenizer to the Kaggle working directory
save_dir = "/kaggle/working/my_mamba_model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

# 2. Zip it so you can download it via the Kaggle right-hand panel
import shutil
shutil.make_archive("my_mamba_model", "zip", save_dir)
print("my_mamba_model.zip is ready for download!")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# Ensure model is in eval mode on GPU
model.eval()

prompt = "Below is an instruction that describes a task.\n\n### Instruction:\nWhat is machine learning?\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
input_ids = inputs["input_ids"]

# Inspect layer 0
layer = model.backbone.layers[0].mixer

# Auto-detect attribute names (HF transformers vs native Mamba)
d_inner = getattr(layer, "intermediate_size", getattr(layer, "d_inner", None))
dt_rank = getattr(layer, "time_step_rank", getattr(layer, "dt_rank", None))
d_state = getattr(layer, "ssm_state_size", getattr(layer, "d_state", None))

with torch.no_grad():
    # 1. Forward through embedding layer
    hidden_states = model.backbone.embeddings(input_ids)
    
    # 2. In-projection (projects hidden_dim -> 2 * d_inner)
    projected = layer.in_proj(hidden_states)
    
    # Split into x branch and residual branch
    x, res = torch.split(projected, [d_inner, d_inner], dim=-1)
    
    # Conv1d operates over [Batch, Channels/Dim, Length]
    x_conv = x.transpose(1, 2)
    x_conv = layer.conv1d(x_conv)[:, :, :input_ids.shape[1]]
    x = F.silu(x_conv).transpose(1, 2)  # Shape: [B, L, D]
    
    # 3. Discretization projections: dt, B, C
    x_dbl = layer.x_proj(x)
    dt, B_proj, C_proj = torch.split(x_dbl, [dt_rank, d_state, d_state], dim=-1)
    
    dt = layer.dt_proj(dt)
    dt = F.softplus(dt)  # Discretization step Δ, Shape: [B, L, D]
    
    # A parameter (stored as log in model)
    A = -torch.exp(layer.A_log.float())  # Shape: [D, N]
    
    # Compute a_t = exp(Δ_t * A)
    # Shape: [B, L, D, N]
    dt_expanded = dt.unsqueeze(-1)           # [B, L, D, 1]
    A_expanded = A.unsqueeze(0).unsqueeze(0) # [1, 1, D, N]
    a_t = torch.exp(dt_expanded * A_expanded)
    
    # Compute b_t = (Δ_t * B_t) * x_t
    # B_proj shape: [B, L, 1, N]
    # x shape:      [B, L, D, 1]
    B_expanded = B_proj.unsqueeze(2)         # [B, L, 1, N]
    x_expanded = x.unsqueeze(-1)             # [B, L, D, 1]
    b_t = (dt_expanded * B_expanded) * x_expanded # [B, L, D, N]

print(f"Extracted Tuples for Sequence Length L = {input_ids.shape[1]}:")
print(f"a_t (State decay factors) shape:     {a_t.shape}") # [1, L, 1536, 16]
print(f"b_t (Input drive projection) shape: {b_t.shape}") # [1, L, 1536, 16]

In [ ]:
def scan_combine(tuple_j, tuple_i):
    """
    Equation (2) from Scan-Fuse:
    tuple_j: (a_j, b_j) - later in time
    tuple_i: (a_i, b_i) - earlier in time
    """
    a_j, b_j = tuple_j
    a_i, b_i = tuple_i
    
    a_out = a_j * a_i
    b_out = a_j * b_i + b_j
    return a_out, b_out

# 1. Sequential Recurrence Baseline
def sequential_scan(a_seq, b_seq):
    L = a_seq.shape[0]
    h = torch.zeros_like(b_seq[0])
    h_out = []
    for t in range(L):
        h = a_seq[t] * h + b_seq[t]
        h_out.append(h)
    return torch.stack(h_out)

# 2. Parallel Associative Scan (Hillis-Steele Tree)
def hillis_steele_scan(a_seq, b_seq):
    L = a_seq.shape[0]
    a_tree = a_seq.clone()
    b_tree = b_seq.clone()
    
    steps = int(np.ceil(np.log2(L)))
    for d in range(steps):
        offset = 1 << d
        # Parallel associative combine for nodes at step d
        a_prev = a_tree[:-offset].clone()
        b_prev = b_tree[:-offset].clone()
        
        a_curr = a_tree[offset:].clone()
        b_curr = b_tree[offset:].clone()
        
        a_comb, b_comb = scan_combine((a_curr, b_curr), (a_prev, b_prev))
        a_tree[offset:] = a_comb
        b_tree[offset:] = b_comb
        
    return b_tree # b_tree now contains the cumulative hidden states h_t

# Verify numerical match
a_data = a_t[0].cpu() # [L, D, N]
b_data = b_t[0].cpu() # [L, D, N]

h_seq = sequential_scan(a_data, b_data)
h_tree = hillis_steele_scan(a_data, b_data)

max_diff = torch.max(torch.abs(h_seq - h_tree)).item()
print(f"Max Absolute Error between Sequential and Associative Tree: {max_diff:.2e}")
assert max_diff < 1e-5, "Scan verification failed!"

In [ ]:
# Select a slice (e.g. sequence length 64, single vector dimension of width 16/32)
L_slice = 64
vector_len = 16 # RVV VLEN/SEW (e.g. 16 half-precision elements)

a_slice = a_data[:L_slice, 0, :vector_len].to(torch.float16).numpy()
b_slice = b_data[:L_slice, 0, :vector_len].to(torch.float16).numpy()
h_expected = h_tree[:L_slice, 0, :vector_len].to(torch.float16).numpy()

with open("test_vectors.h", "w") as f:
    f.write("// Auto-generated from fine-tuned Mamba-130M model\n")
    f.write(f"#define SEQ_LEN {L_slice}\n")
    f.write(f"#define VEC_LEN {vector_len}\n\n")
    
    def dump_array(name, arr):
        # Dump as 16-bit raw hex values
        f.write(f"const uint16_t {name}[SEQ_LEN][VEC_LEN] = {{\n")
        raw = arr.view(np.uint16)
        for row in raw:
            f.write("    {" + ", ".join([f"0x{val:04x}" for val in row]) + "},\n")
        f.write("};\n\n")
        
    dump_array("a_tuples", a_slice)
    dump_array("b_tuples", b_slice)
    dump_array("golden_h", h_expected)

print("Saved test_vectors.h for RISC-V testbench!")

In [ ]:
 [code] {"execution":{"iopub.status.busy":"2026-10-12T07:50:09.622020Z","iopub.execute_input":"2026-10-12T07:50:09.622456Z","iopub.status.idle":"2026-10-12T07:50:09.633678Z","shell.execute_reply.started":"2026-10-12T07:50:09.622424Z","shell.execute_reply":"2026-10-12T07:50:09.632970Z"}}

In [ ]:
import numpy as np
import torch

# 1. Take a 4-step sequence from Layer 0 of your trained model (channel 0)
# a_data is State Decay (P), b_data is Input Projection (Q)
p_real = a_data[:4, 0, 0].numpy()  # P_0, P_1, P_2, P_3
q_real = b_data[:4, 0, 0].numpy()  # Q_0, Q_1, Q_2, Q_3

# 2. Quantize to INT8 (matching Mamba-X H2 Quantization)
# P represents decay between 0.0 and 1.0 -> scale by 64 (Q2.6 fixed-point)
# Q represents input drive -> scale by 32
scale_p = 64.0
scale_q = 32.0

P_int8 = np.clip(np.round(p_real * scale_p), -128, 127).astype(np.int8)
Q_int8 = np.clip(np.round(q_real * scale_q), -128, 127).astype(np.int8)

# 3. Calculate Golden State using the exact Verilog hardware arithmetic
# Hardware formula: state_n = ((P_n * state_{n-1}) >> 6) + Q_n
golden_states = []
state = int(Q_int8[0]) # state_0 = Q_0
golden_states.append(state)

for i in range(1, 4):
    p_val = int(P_int8[i])
    q_val = int(Q_int8[i])
    # Bit shift by 6 matches SHIFT_VAL = 6 in spe.v
    state = ((p_val * state) >> 6) + q_val
    state = int(np.clip(state, -128, 127))
    golden_states.append(state)

# 4. Save to pure .hex files for Verilog $readmemh
def write_verilog_hex(filename, data):
    with open(f"/kaggle/working/{filename}", "w") as f:
        for val in data:
            # Format as 2-digit unsigned hex byte (e.g., 0a, 20, ff)
            f.write(f"{(int(val) & 0xFF):02x}\n")

write_verilog_hex("inputs_P.hex", P_int8)
write_verilog_hex("inputs_Q.hex", Q_int8)
write_verilog_hex("golden_state.hex", golden_states)

print("=" * 45)
print("REAL MAMBA MODEL TEST VECTORS GENERATED!")
print("=" * 45)
print("P values (Decay):", [int(x) for x in P_int8])
print("Q values (Input):", [int(x) for x in Q_int8])
print("Golden States:   ", golden_states)
print("\nHex files saved to /kaggle/working/:")
print("- inputs_P.hex")
print("- inputs_Q.hex")
print("- golden_state.hex")